# Multi-Agent Loan Underwriting - demo
Run all cells from top to bottom using the project `.venv` kernel.

In [1]:
from pathlib import Path
import os
import tempfile
from dotenv import load_dotenv

project_root = Path.cwd().resolve()
while not (project_root / "loan_uw").is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the loan_underwriting project root")
    project_root = project_root.parent
os.chdir(project_root)
load_dotenv(project_root / ".env")

from loan_uw.agents.underwriting_manager import UnderwritingManager
from loan_uw.config import get_groq_api_key, inr
from loan_uw.runlog import AppendOnlyRunLog
from loan_uw.samples import SAMPLES
from loan_uw.trace import format_event

print("extractor: " + ("groq" if get_groq_api_key() else "regex"))
log_file = tempfile.NamedTemporaryFile(prefix="loan-underwriting-demo-", suffix=".jsonl", delete=False)
log_file.close()
log_path = Path(log_file.name)
log = AppendOnlyRunLog(log_path)

async def show_scenario(name):
    manager = UnderwritingManager(log)
    record = await manager.process(SAMPLES[name])
    for event in manager.last_trace:
        print(format_event(event))
    print("Decision:", record.decision.value.upper())
    print("EMI:", inr(record.emi) if record.emi is not None else "not computed (halted)")
    if record.affordability is None:
        print("Affordability: not evaluated")
    else:
        verdict = "PASS" if record.affordability.fits else "FAIL"
        print(f"Affordability: {verdict} ({record.affordability.obligation_ratio:.1%} vs {record.affordability.cap:.0%})")
    if record.counter_offers:
        for offer in record.counter_offers:
            print(f"Counter-offer: {offer.kind}: {inr(offer.amount)} / {offer.tenure_months} months / EMI {inr(offer.emi)}")
    else:
        print("Counter-offers: none")
    print("Justification:", record.justification)
    return record

extractor: groq


## Approve
Demonstrates a clean application that meets the affordability and credit rules and is sanctioned.

In [2]:
await show_scenario("approve")

[  1] UnderwritingManager  RUN_START  {"raw_text": "Applicant Ananya Sharma (APP-1001) requests a personal loan of Rs 3,00,000 for 36 months at 11% interest. Net mon...
[  2] UnderwritingManager  DELEGATE   -> DocumentAgent: extract income, existing EMIs, amount, tenure
[  3] DocumentAgent        READ       fields=raw_text
[  4] DocumentAgent        TOOL_CALL  document.llm_extract_fields({"text": "Applicant Ananya Sharma (APP-1001) requests a personal lo...) -> {"monthly_net_income": 90000, "existing_emis": 8000, "requested_amount": 300000, "tenur...
[  5] DocumentAgent        EXTRACTOR_USED {"extractor": "groq", "fallback_reason": null}
[  6] DocumentAgent        TOOL_CALL  document.validate_fields({"fields": {"monthly_net_income": 90000, "existing_emis": 8000, "re...) -> {"missing": []}
[  7] DocumentAgent        WRITE      fields={"applicant_id": "APP-1001", "applicant_name": "Ananya Sharma", "monthly_net_income": 90000, "existing_emis...
[  8] DocumentAgent        REPORT     OK | i

ApplicantRecord(application_id='LN-3E7DAC47', raw_text='Applicant Ananya Sharma (APP-1001) requests a personal loan of Rs 3,00,000 for 36 months at 11% interest. Net monthly income Rs 90,000. Existing EMIs of Rs 8,000.', applicant_id='APP-1001', applicant_name='Ananya Sharma', monthly_net_income=90000.0, existing_emis=8000.0, requested_amount=300000.0, tenure_months=36, annual_rate_pct=11.0, missing_fields=[], assumptions=[], emi=9821.62, total_payable=353578.32, total_interest=53578.32, affordability=AffordabilityResult(fits=True, obligation_ratio=0.198, cap=0.4, max_affordable_emi=28000.0, rule='EMI + existing EMIs <= 40% of net monthly income'), credit=CreditResult(record_found=True, credit_score=782, flags=[]), counter_offers=[], decision=<Decision.SANCTION: 'sanction'>, justification='Decision: SANCTION (rule R5: all checks passed -> SANCTION). Requested Rs 300,000.00 over 36 months at 11.0% gives a reducing-balance EMI of Rs 9,821.62; with existing EMIs of Rs 8,000.00, total obli

## Decline
Demonstrates an affordable application declined because of a hard credit-history rule.

In [3]:
await show_scenario("decline")

[  1] UnderwritingManager  RUN_START  {"raw_text": "Applicant Karan Mehta (APP-1002) requests a personal loan of Rs 2,00,000 for 24 months. Net monthly income Rs 70,...
[  2] UnderwritingManager  DELEGATE   -> DocumentAgent: extract income, existing EMIs, amount, tenure
[  3] DocumentAgent        READ       fields=raw_text
[  4] DocumentAgent        TOOL_CALL  document.llm_extract_fields({"text": "Applicant Karan Mehta (APP-1002) requests a personal loan...) -> {"monthly_net_income": 70000, "existing_emis": 4000, "requested_amount": 200000, "tenur...
[  5] DocumentAgent        EXTRACTOR_USED {"extractor": "groq", "fallback_reason": null}
[  6] DocumentAgent        TOOL_CALL  document.validate_fields({"fields": {"monthly_net_income": 70000, "existing_emis": 4000, "re...) -> {"missing": []}
[  7] DocumentAgent        WRITE      fields={"applicant_id": "APP-1002", "applicant_name": "Karan Mehta", "monthly_net_income": 70000, "existing_emis":...
[  8] DocumentAgent        REPORT     OK | i

ApplicantRecord(application_id='LN-79B93DC1', raw_text='Applicant Karan Mehta (APP-1002) requests a personal loan of Rs 2,00,000 for 24 months. Net monthly income Rs 70,000. Existing EMIs of Rs 4,000.', applicant_id='APP-1002', applicant_name='Karan Mehta', monthly_net_income=70000.0, existing_emis=4000.0, requested_amount=200000.0, tenure_months=24, annual_rate_pct=10.5, missing_fields=[], assumptions=['interest rate not stated; bank default 10.5% used'], emi=9275.21, total_payable=222605.04, total_interest=22605.04, affordability=AffordabilityResult(fits=True, obligation_ratio=0.1896, cap=0.4, max_affordable_emi=24000.0, rule='EMI + existing EMIs <= 40% of net monthly income'), credit=CreditResult(record_found=True, credit_score=540, flags=[RedFlag(code='DEFAULT', severity='hard', reason='1 prior default(s) on record'), RedFlag(code='LOW_SCORE', severity='hard', reason='credit score 540 below minimum 600'), RedFlag(code='LATE_PAYMENTS', severity='review', reason='5 late payments in t

## Counter-offer
Demonstrates an unaffordable request declined with independently verified alternatives.

In [4]:
await show_scenario("counter_offer")

[  1] UnderwritingManager  RUN_START  {"raw_text": "Applicant Priya Nair (APP-1003) requests a loan of Rs 10,00,000 for 36 months. Net monthly income Rs 60,000. Exis...
[  2] UnderwritingManager  DELEGATE   -> DocumentAgent: extract income, existing EMIs, amount, tenure
[  3] DocumentAgent        READ       fields=raw_text
[  4] DocumentAgent        TOOL_CALL  document.llm_extract_fields({"text": "Applicant Priya Nair (APP-1003) requests a loan of Rs 10,...) -> {"monthly_net_income": 60000, "existing_emis": 5000, "requested_amount": 1000000, "tenu...
[  5] DocumentAgent        EXTRACTOR_USED {"extractor": "groq", "fallback_reason": null}
[  6] DocumentAgent        TOOL_CALL  document.validate_fields({"fields": {"monthly_net_income": 60000, "existing_emis": 5000, "re...) -> {"missing": []}
[  7] DocumentAgent        WRITE      fields={"applicant_id": "APP-1003", "applicant_name": "Priya Nair", "monthly_net_income": 60000, "existing_emis": ...
[  8] DocumentAgent        REPORT     OK | i

ApplicantRecord(application_id='LN-953B8489', raw_text='Applicant Priya Nair (APP-1003) requests a loan of Rs 10,00,000 for 36 months. Net monthly income Rs 60,000. Existing EMIs of Rs 5,000.', applicant_id='APP-1003', applicant_name='Priya Nair', monthly_net_income=60000.0, existing_emis=5000.0, requested_amount=1000000.0, tenure_months=36, annual_rate_pct=10.5, missing_fields=[], assumptions=['interest rate not stated; bank default 10.5% used'], emi=32502.44, total_payable=1170087.84, total_interest=170087.84, affordability=AffordabilityResult(fits=False, obligation_ratio=0.625, cap=0.4, max_affordable_emi=19000.0, rule='EMI + existing EMIs <= 40% of net monthly income'), credit=CreditResult(record_found=True, credit_score=741, flags=[]), counter_offers=[CounterOffer(kind='reduce_amount', amount=584000.0, tenure_months=36, emi=18981.43, obligation_ratio=0.3997), CounterOffer(kind='extend_tenure', amount=1000000.0, tenure_months=84, emi=16860.67, obligation_ratio=0.3643)], decision=<D

## Missing income
Demonstrates that missing income halts calculations and refers the application.

In [5]:
await show_scenario("missing_income")

[  1] UnderwritingManager  RUN_START  {"raw_text": "Applicant Rohan Verma (APP-1004) requests a loan of Rs 2,00,000 for 24 months."}
[  2] UnderwritingManager  DELEGATE   -> DocumentAgent: extract income, existing EMIs, amount, tenure
[  3] DocumentAgent        READ       fields=raw_text
[  4] DocumentAgent        TOOL_CALL  document.llm_extract_fields({"text": "Applicant Rohan Verma (APP-1004) requests a loan of Rs 2,...) -> {"monthly_net_income": null, "existing_emis": null, "requested_amount": 200000, "tenure...
[  5] DocumentAgent        EXTRACTOR_USED {"extractor": "groq", "fallback_reason": null}
[  6] DocumentAgent        TOOL_CALL  document.validate_fields({"fields": {"monthly_net_income": null, "existing_emis": null, "req...) -> {"missing": ["monthly_net_income"]}
[  7] DocumentAgent        WRITE      fields={"applicant_id": "APP-1004", "applicant_name": "Rohan Verma", "monthly_net_income": null, "existing_emis": ...
[  8] DocumentAgent        REPORT     FAIL | missing mandato

ApplicantRecord(application_id='LN-33C7C6C0', raw_text='Applicant Rohan Verma (APP-1004) requests a loan of Rs 2,00,000 for 24 months.', applicant_id='APP-1004', applicant_name='Rohan Verma', monthly_net_income=None, existing_emis=0.0, requested_amount=200000.0, tenure_months=24, annual_rate_pct=10.5, missing_fields=['monthly_net_income'], assumptions=['existing EMIs not stated; treated as Rs 0', 'interest rate not stated; bank default 10.5% used'], emi=None, total_payable=None, total_interest=None, affordability=None, credit=None, counter_offers=[], decision=<Decision.REFER: 'refer'>, justification='Decision: REFER (rule R1: mandatory data missing -> REFER (never assume)). The application cannot be assessed because the following mandatory input is missing: monthly_net_income; it was not assumed or estimated. No EMI, affordability or credit checks were run. A human underwriter must obtain the missing information.', status='halted_missing_data')

## Credit review
Demonstrates a referral when the credit history triggers review rather than decline.

In [6]:
await show_scenario("credit_review")

[  1] UnderwritingManager  RUN_START  {"raw_text": "Applicant Sana Iqbal (APP-1005) requests a loan of Rs 1,50,000 for 24 months. Net monthly income Rs 80,000. Exist...
[  2] UnderwritingManager  DELEGATE   -> DocumentAgent: extract income, existing EMIs, amount, tenure
[  3] DocumentAgent        READ       fields=raw_text
[  4] DocumentAgent        TOOL_CALL  document.llm_extract_fields({"text": "Applicant Sana Iqbal (APP-1005) requests a loan of Rs 1,5...) -> {"monthly_net_income": 80000, "existing_emis": 2000, "requested_amount": 150000, "tenur...
[  5] DocumentAgent        EXTRACTOR_USED {"extractor": "groq", "fallback_reason": null}
[  6] DocumentAgent        TOOL_CALL  document.validate_fields({"fields": {"monthly_net_income": 80000, "existing_emis": 2000, "re...) -> {"missing": []}
[  7] DocumentAgent        WRITE      fields={"applicant_id": "APP-1005", "applicant_name": "Sana Iqbal", "monthly_net_income": 80000, "existing_emis": ...
[  8] DocumentAgent        REPORT     OK | i

ApplicantRecord(application_id='LN-36F9936A', raw_text='Applicant Sana Iqbal (APP-1005) requests a loan of Rs 1,50,000 for 24 months. Net monthly income Rs 80,000. Existing EMIs of Rs 2,000.', applicant_id='APP-1005', applicant_name='Sana Iqbal', monthly_net_income=80000.0, existing_emis=2000.0, requested_amount=150000.0, tenure_months=24, annual_rate_pct=10.5, missing_fields=[], assumptions=['interest rate not stated; bank default 10.5% used'], emi=6956.41, total_payable=166953.84, total_interest=16953.84, affordability=AffordabilityResult(fits=True, obligation_ratio=0.112, cap=0.4, max_affordable_emi=30000.0, rule='EMI + existing EMIs <= 40% of net monthly income'), credit=CreditResult(record_found=True, credit_score=688, flags=[RedFlag(code='LATE_PAYMENTS', severity='review', reason='3 late payments in the last 12 months')]), counter_offers=[], decision=<Decision.REFER: 'refer'>, justification='Decision: REFER (rule R3: credit review flag -> REFER to human underwriter). Requested Rs

## Run-log verification
Verify that all scenario events in this notebook's temporary append-only log form an intact hash chain.

In [7]:
print("run-log hash chain intact:", log.verify()[0])
log_path.unlink(missing_ok=True)

run-log hash chain intact: True
